# Appendix C: Fourier Transform of the Squared Envelope $\Gamma(t) = \alpha^2(t)$

This notebook derives $\hat{\Gamma}(\omega)$ where $\Gamma(t) = \alpha^2(t)$ is the squared
trapezoidal envelope. The ramps become parabolic (concave) when squared.

**Key result:**
$$\hat{\Gamma}(\omega) = \frac{4\alpha_{\max}^2}{\tau^2\omega^3}\left[\tau\omega\cos\!\left(\frac{\omega\ell}{2}\right) + \sin\!\left(\frac{\omega\ell}{2}\right) - \sin\!\left(\frac{\omega\ell}{2} + \omega\tau\right)\right]$$

**Normalization.** The paper (main-text Eq. `Gamma_hat`, Appendix `app:fourier_trapezoid`, Eqs. `Gamma_piecewise`,
`Gamma_hat_direct`) transforms the *normalized* envelope $\Gamma(t) = \alpha^2(t)/\alpha_{\max}^2$, i.e. the result
above divided by $\alpha_{\max}^2$, and writes $\ell \equiv \ell_{\rm spot}$, $\tau \equiv \tau_{\rm spot}$. The checks below
compare against those printed equations, taken from the equation registry (`equations.py`), where each is
transcribed into SymPy once, exactly as typeset; `check_equations.py` keeps the transcriptions in sync with `ms.tex`.

In [ ]:
import sympy as sp
from sympy import pi, cos, sin, integrate, simplify, symbols, latex
from IPython.display import display, Math
sp.init_printing()

In [ ]:
t = sp.Symbol('t', real=True)
w = sp.Symbol('omega', positive=True)
ell = sp.Symbol('ell', positive=True)
tau = sp.Symbol('tau', positive=True)
amax = sp.Symbol('alpha_max', positive=True)

## Piecewise definition of $\Gamma(t)$

Squaring the trapezoid:
$$\Gamma(t) = \alpha^2(t) = \alpha_{\max}^2 \times \begin{cases} 1 & |t| \leq \ell/2 \\ \left(\frac{\ell/2 + \tau - |t|}{\tau}\right)^2 & \ell/2 < |t| \leq \ell/2 + \tau \\ 0 & \text{otherwise} \end{cases}$$

The ramps are now **parabolic** rather than linear.

### Check against the paper: piecewise definitions (Eqs. `alpha_piecewise`, `Gamma_piecewise`)

The paper's five-branch trapezoid $\alpha(t)$ and its normalized square $\Gamma(t)$ are taken from the registry
($\ell_{\rm spot} \to \ell$, $\tau_{\rm spot} \to \tau$). Checked: on each ramp $\Gamma = (\alpha/\alpha_{\max})^2$, $\alpha$ is even
(which the cosine-transform shortcut below relies on), $\alpha$ is continuous at the breakpoints, and the two
`Piecewise` expressions satisfy $\Gamma = (\alpha/\alpha_{\max})^2$ at every point of a grid that crosses all branches.

In [ ]:
import equations as eqs   # the paper's equations, transcribed once (src/derivations/equations.py)

to_nb = {eqs.tau_s: tau}   # tau_spot is tau here; t, omega, ell and alpha_max are the registry's own symbols

alpha_paper = eqs.get("eq:alpha_piecewise").expr.subs(to_nb)   # Eq. alpha_piecewise
Gamma_paper = eqs.get("eq:Gamma_piecewise").expr.subs(to_nb)   # Eq. Gamma_piecewise (normalized by alpha_max^2)

# The rising and falling ramps of alpha, in units of alpha_max
alpha_branches = [e for e, _ in (alpha_paper / amax).args]
assert len(alpha_branches) == 5
alpha_rise_paper = alpha_branches[1]   # -ell/2 - tau <= t < -ell/2
alpha_fall_paper = alpha_branches[3]   #  ell/2 < t <= ell/2 + tau

# The ramp of Gamma, as a function of |t|
(Gamma_ramp_abs,) = [e for e, _ in Gamma_paper.args if e.has(sp.Abs(t))]


def Gamma_ramp_paper(abs_t):
    # Ramp branch of Eq. Gamma_piecewise as a function of |t|
    return Gamma_ramp_abs.subs(sp.Abs(t), abs_t)


assert simplify(Gamma_ramp_paper(t) - alpha_fall_paper**2) == 0      # t > 0: |t| = t
assert simplify(Gamma_ramp_paper(-t) - alpha_rise_paper**2) == 0     # t < 0: |t| = -t
assert simplify(alpha_rise_paper.subs(t, -t) - alpha_fall_paper) == 0  # alpha is even
assert simplify(alpha_rise_paper.subs(t, -ell / 2 - tau)) == 0 and simplify(alpha_rise_paper.subs(t, -ell / 2)) == 1
assert simplify(alpha_fall_paper.subs(t, ell / 2)) == 1 and simplify(alpha_fall_paper.subs(t, ell / 2 + tau)) == 0

pars = {ell: 6, tau: 2}                          # breakpoints at |t| = 3, 5
for k in range(-28, 29):
    tt = sp.Rational(k, 4)
    a_val = alpha_paper.subs(pars).subs(t, tt) / amax
    assert Gamma_paper.subs(pars).subs(t, tt) == a_val**2, f"Gamma != (alpha/alpha_max)^2 at t = {tt}"
print("Verified: Eq. Gamma_piecewise = (Eq. alpha_piecewise / alpha_max)^2; alpha even and continuous")

## Direct integration

Since $\Gamma(t)$ is real and even:
$$\hat{\Gamma}(\omega) = 2\alpha_{\max}^2\left[\underbrace{\int_0^{\ell/2}\cos(\omega t)\,dt}_{I_1} + \underbrace{\int_{\ell/2}^{\ell/2+\tau}\left(\frac{\ell/2 + \tau - t}{\tau}\right)^2\cos(\omega t)\,dt}_{I_2}\right]$$

In [ ]:
# I_1: plateau integral
I1 = integrate(cos(w * t), (t, 0, ell / 2))
I1 = simplify(I1)
print("I_1 (plateau):")
display(Math(rf"I_1 = {latex(I1)}"))

In [ ]:
# I_2: parabolic ramp integral
ramp_sq = ((ell / 2 + tau - t) / tau)**2
I2 = integrate(ramp_sq * cos(w * t), (t, ell / 2, ell / 2 + tau))
I2 = simplify(I2)
print("I_2 (parabolic ramp):")
display(Math(rf"I_2 = {latex(I2)}"))

In [ ]:
# Verify that sin(w*ell/2)/w terms cancel between I1 and I2
I1_plus_I2 = simplify(I1 + I2)
print("I_1 + I_2:")
display(Math(rf"I_1 + I_2 = {latex(I1_plus_I2)}"))

### Check against the paper: $I_1$, $I_2$ and $I_1 + I_2$ (Appendix `app:fourier_trapezoid`, Direct Integration)

The integrand of $I_2$ above must be the $t > 0$ ramp of Eq. `Gamma_piecewise`. The paper's $I_1$ (Eq. `I1_plateau`), the
inner integral $J$ of the second integration by parts (Eq. `J_ibp`), $I_2$ (Eq. `I2_ramp`) and $I_1 + I_2$ after the
$\sin(\omega\ell/2)/\omega$ cancellation (Eq. `I1_plus_I2`) are taken from the registry, and the boundary term of the
first integration by parts, which is printed inline, is transcribed here. All are compared with the derived integrals.

In [ ]:
assert simplify(ramp_sq - Gamma_ramp_paper(t)) == 0, "ramp used for I_2 != Eq. Gamma_piecewise"

a_lo, b_hi = ell / 2, ell / 2 + tau
f_ramp = ((b_hi - t) / tau)**2
boundary = (f_ramp * sin(w * t) / w).subs(t, b_hi) - (f_ramp * sin(w * t) / w).subs(t, a_lo)
J = integrate((b_hi - t) * sin(w * t), (t, a_lo, b_hi))

I1_paper = eqs.get("eq:I1_plateau").expr.subs(to_nb)
boundary_paper = -sin(w * ell / 2) / w                  # printed inline after the first integration by parts
J_paper = eqs.get("eq:J_ibp").expr.subs(to_nb)
I2_paper = eqs.get("eq:I2_ramp").expr.subs(to_nb)
I12_paper = eqs.get("eq:I1_plus_I2").expr.subs(to_nb)

assert simplify(I1 - I1_paper) == 0, "I_1 != Eq. I1_plateau"
assert simplify(f_ramp.subs(t, a_lo)) == 1 and simplify(boundary - boundary_paper) == 0
assert simplify(J - J_paper) == 0, "J != Eq. J_ibp"
assert simplify(boundary + 2 / (tau**2 * w) * J - I2) == 0, "first integration by parts != I_2"
assert simplify(I2 - I2_paper) == 0, "I_2 != Eq. I2_ramp"
assert simplify(I1 + I2 - I12_paper) == 0, "I_1 + I_2 != Eq. I1_plus_I2"
print("Verified: Eqs. I1_plateau, J_ibp, I2_ramp, I1_plus_I2 and the boundary term of Appendix app:fourier_trapezoid")

In [ ]:
# Full Gamma_hat
Gamma_hat = 2 * amax**2 * (I1 + I2)
Gamma_hat = simplify(Gamma_hat)
print("Gamma_hat(omega) for the unnormalized alpha^2 (alpha_max^2 x Eq. Gamma_hat_direct):")
display(Math(rf"\hat{{\Gamma}}(\omega) = {latex(Gamma_hat)}"))

# Verify against expected form
expected = 4 * amax**2 / (tau**2 * w**3) * (
    tau * w * cos(w * ell / 2)
    + sin(w * ell / 2)
    - sin(w * ell / 2 + w * tau)
)
assert simplify(Gamma_hat - expected) == 0
print("\nVerified: matches expected form.")

### Check against the paper: Eq. `Gamma_hat` (main text) and Eq. `Gamma_hat_direct` (appendix)

Both are the transform of the normalized $\Gamma = \alpha^2/\alpha_{\max}^2$, so `Gamma_hat / alpha_max**2` must equal
Eq. `Gamma_hat` (written in $\ell_{\rm spot}$, $\tau_{\rm spot}$) and Eq. `Gamma_hat_direct` (written in $\ell$, $\tau$). As an
independent route that does not use the evenness shortcut, the full complex transform
$\int\Gamma(t)\,e^{-i\omega t}\,dt$ of the paper's Eq. `Gamma_piecewise` (rising ramp, plateau, falling ramp) must equal
the same expression.

In [ ]:
Gamma_hat_eq = eqs.get("eq:Gamma_hat").expr.subs(to_nb)                 # Eq. Gamma_hat (ell_spot, tau_spot)
Gamma_hat_direct_eq = eqs.get("eq:Gamma_hat_direct").expr.subs(to_nb)   # Eq. Gamma_hat_direct (ell, tau)

Gamma_hat_norm = Gamma_hat / amax**2
assert simplify(Gamma_hat_norm - Gamma_hat_eq) == 0, "!= Eq. Gamma_hat"
assert simplify(Gamma_hat_norm - Gamma_hat_direct_eq) == 0, "!= Eq. Gamma_hat_direct"

phase = sp.exp(-sp.I * w * t)
Gamma_hat_full = (integrate(Gamma_ramp_paper(-t) * phase, (t, -ell / 2 - tau, -ell / 2))
                  + integrate(phase, (t, -ell / 2, ell / 2))
                  + integrate(Gamma_ramp_paper(t) * phase, (t, ell / 2, ell / 2 + tau)))
assert simplify((Gamma_hat_full - Gamma_hat_direct_eq).rewrite(sp.exp)) == 0, "complex FT of Eq. Gamma_piecewise != Eq. Gamma_hat_direct"
print("Verified: Gamma_hat / alpha_max^2 = Eq. Gamma_hat = Eq. Gamma_hat_direct = full transform of Eq. Gamma_piecewise")

## DC limit

Expand the bracket to third order in $\omega$ to evaluate $\hat{\Gamma}(0)$.

In [ ]:
# DC limit via Taylor expansion
dc_limit = sp.limit(Gamma_hat, w, 0)
print(f"DC limit: Gamma_hat(0) = {dc_limit}")

# Verify: area under Gamma = amax^2 * (ell + 2*tau/3)
# Plateau: amax^2 * ell
# Each parabolic ramp: amax^2 * int_0^1 u^2 du * tau = amax^2 * tau/3
expected_area = amax**2 * (ell + 2 * tau / 3)
assert simplify(dc_limit - expected_area) == 0
print(f"Verified: equals area under Gamma = alpha_max^2 * (ell + 2*tau/3)")

In [ ]:
# Show the Taylor expansion explicitly (as done in the appendix)
bracket = tau * w * cos(w * ell / 2) + sin(w * ell / 2) - sin(w * ell / 2 + w * tau)
bracket_series = sp.series(bracket, w, 0, n=6)
print("Taylor expansion of bracket term:")
display(Math(rf"B(\omega) = {latex(bracket_series)}"))
bracket_poly = bracket_series.removeO()
assert all(simplify(bracket_poly.coeff(w, k)) == 0 for k in range(3)), "B(omega) is not O(omega^3)"
assert simplify(bracket_poly.coeff(w, 3) - tau**2 * (ell + 2 * tau / 3) / 4) == 0
print("\nThe w^1 terms cancel, leaving w^3 as leading order.")
print("Dividing by w^3 gives the finite DC limit.")

In [ ]:
# Numerical verification
import numpy as np

def Gamma_numpy(t_arr, ell_n, tau_n, amax_n):
    abs_t = np.abs(t_arr)
    L = ell_n / 2
    S = L + tau_n
    result = np.zeros_like(abs_t)
    plateau = abs_t <= L
    ramp = (abs_t > L) & (abs_t <= S)
    result[plateau] = amax_n**2
    result[ramp] = amax_n**2 * ((S - abs_t[ramp]) / tau_n)**2
    return result

def Gamma_hat_numerical(w_arr, ell_n, tau_n, amax_n, nt=20000):
    S = ell_n / 2 + tau_n
    t_grid = np.linspace(-S, S, nt)
    G_t = Gamma_numpy(t_grid, ell_n, tau_n, amax_n)
    return np.array([np.trapezoid(G_t * np.cos(wv * t_grid), t_grid) for wv in w_arr])

def Gamma_hat_analytic(w_arr, ell_n, tau_n, amax_n):
    return 4 * amax_n**2 / (tau_n**2 * w_arr**3) * (
        tau_n * w_arr * np.cos(w_arr * ell_n / 2)
        + np.sin(w_arr * ell_n / 2)
        - np.sin(w_arr * ell_n / 2 + w_arr * tau_n))

ell_n, tau_n, amax_n = 15.0, 3.0, 0.1
w_arr = np.linspace(0.01, 4.0, 500)

Gh_analytic = Gamma_hat_analytic(w_arr, ell_n, tau_n, amax_n)
Gh_numerical = Gamma_hat_numerical(w_arr, ell_n, tau_n, amax_n)

max_err = np.max(np.abs(Gh_analytic - Gh_numerical))
print(f"Max error (analytic vs numerical): {max_err:.2e}")
assert max_err < 1e-4, "analytic Gamma_hat disagrees with numerical integration"
print("Numerical verification passed.")

### Numerical check of the paper's equations

The same quadrature applied to the paper's own (normalized) Eq. `Gamma_piecewise`, compared with the paper's
Eq. `Gamma_hat`, both lambdified from the registry expressions above.

In [ ]:
Gamma_paper_np = sp.lambdify((t, ell, tau), Gamma_paper, "numpy")
Gamma_hat_eq_np = sp.lambdify((w, ell, tau), Gamma_hat_eq, "numpy")

S_n = ell_n / 2 + tau_n
t_grid = np.linspace(-S_n, S_n, 20000)
G_paper_t = Gamma_paper_np(t_grid, ell_n, tau_n)
assert np.allclose(G_paper_t, Gamma_numpy(t_grid, ell_n, tau_n, amax_n) / amax_n**2, rtol=0, atol=1e-14)

Gh_num_paper = np.array([np.trapezoid(G_paper_t * np.cos(wv * t_grid), t_grid) for wv in w_arr])
Gh_eq = Gamma_hat_eq_np(w_arr, ell_n, tau_n)
rel_err = np.max(np.abs(Gh_num_paper - Gh_eq)) / np.max(np.abs(Gh_eq))
print(f"Max relative error (Eq. Gamma_hat vs quadrature of Eq. Gamma_piecewise): {rel_err:.2e}")
assert rel_err < 1e-6, "Eq. Gamma_hat disagrees with numerical integration of Eq. Gamma_piecewise"